In [ ]:
import os
import ast
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, binomtest
from statsmodels.stats.multitest import multipletests
from sklearn.metrics import brier_score_loss, balanced_accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, roc_auc_score,  mean_absolute_error, mean_squared_error

Working with Seeds

Seed changes:
* weight initialization
* batch order
* dropout masks

Merging files

* Comparings the best perfomance setting of Cola results to the Baselines' results (GRU + Trans) 

In [ ]:
task = "REG"  # "REG" or "BIN"

In [ ]:
seed_files = {
    "COLA_Alex_LogMel_Trait": {
        42: f"/workspace/app/planilhas/TaCoLa/Seeds/{task}_COLA_Alex_LogMel_Trait_seed42.xlsx",
        123: f"/workspace/app/planilhas/TaCoLa/Seeds/{task}_COLA_Alex_LogMel_Trait_seed123.xlsx",
        2026: f"/workspace/app/planilhas/TaCoLa/Seeds/{task}_COLA_Alex_LogMel_Trait_seed2026.xlsx"},
    #"COLA_Speech": {
    #        42: f"/workspace/app/planilhas/TaCoLa/Seeds/{task}_COLA_Speech_seed42.xlsx",
    #        123: f"/workspace/app/planilhas/TaCoLa/Seeds/{task}_COLA_Speech_seed123.xlsx",
    #        2026: f"/workspace/app/planilhas/TaCoLa/Seeds/{task}_COLA_Speech_seed2026.xlsx"},
    "GRU_LogMel": {
        42: f"/workspace/app/planilhas/TaCoLa/Seeds/{task}_GRU_LogMel_seed42.xlsx",
        123: f"/workspace/app/planilhas/TaCoLa/Seeds/{task}_GRU_LogMel_seed123.xlsx",
        2026: f"/workspace/app/planilhas/TaCoLa/Seeds/{task}_GRU_LogMel_seed2026.xlsx"},
    "TRANS_LogMel": {
        42: f"/workspace/app/planilhas/TaCoLa/Seeds/{task}_TRANS_LogMel_seed42.xlsx",
        123: f"/workspace/app/planilhas/TaCoLa/Seeds/{task}_TRANS_LogMel_seed123.xlsx",
        2026: f"/workspace/app/planilhas/TaCoLa/Seeds/{task}_TRANS_LogMel_seed2026.xlsx"}
}
all_seeds = []
for model, files in seed_files.items():
    for seed, path in files.items():
        d = pd.read_excel(path)
        d["Experiment"] = model
        d["Seed"] = seed
        all_seeds.append(d)
df_seeds = pd.concat(all_seeds, ignore_index=True)

In [ ]:
def scalar_from_list(value):
    if isinstance(value, (list, tuple, np.ndarray)):
        return float(np.asarray(value).reshape(-1)[0])
    if isinstance(value, str):
        parsed = ast.literal_eval(value)
        if isinstance(parsed, (list, tuple, np.ndarray)):
            return float(np.asarray(parsed).reshape(-1)[0])
        return float(parsed)
    return float(value)

if task == "BIN":
    required = ["Patient_ID", "y_patient_true", "p_patient"]
    for col in required:
        df_seeds[col] = pd.to_numeric(df_seeds[col], errors="coerce")
else:
    required = ["Patient_ID", "y_true", "y_pred"]
    for col in ["y_true", "y_pred"]:
        df_seeds[col] = df_seeds[col].apply(scalar_from_list)
        df_seeds[col] = pd.to_numeric(df_seeds[col], errors="coerce")

invalid = df_seeds[df_seeds[required].isna().any(axis=1)].copy()
if not invalid.empty:
    print("Removing invalid folds:")
    print(invalid.groupby(["Experiment", "Seed"]).size())

df_seeds = df_seeds.dropna(subset=required).copy()

counts = df_seeds.groupby(["Experiment", "Seed"])["Patient_ID"].nunique()
print(counts)
if task == "BIN" and not (counts == 50).all():
    raise ValueError("Every model/seed must contain 50 valid patients.")

duplicates = df_seeds.duplicated(["Experiment", "Seed", "Patient_ID"])
if duplicates.any():
    raise ValueError("Duplicated patient predictions were found.")

In [ ]:
patient_sets = [
    set(d["Patient_ID"])
    for _, d in df_seeds.groupby(["Experiment", "Seed"])
]

if not all(patients == patient_sets[0] for patients in patient_sets[1:]):
    raise ValueError("Model/seed combinations do not contain the same patients.")

print("Validation passed: all model/seed combinations contain the same number of patients.")

In [ ]:
model_ref = "COLA_Alex_LogMel_Trait"
#model_ref = "COLA_Speech"
baselines = ["GRU_LogMel", "TRANS_LogMel"]

___

Metrics

In [ ]:
def binary_metrics(d, tau=0.5):
    y = d["y_patient_true"].astype(int).to_numpy()
    p = d["p_patient"].astype(float).to_numpy()
    pred = (p >= tau).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {
        "Balanced_Accuracy": balanced_accuracy_score(y, pred),
        #"Macro_F1": f1_score(y, pred, average="macro", zero_division=0),
        "Sensitivity": tp / (tp + fn) if tp + fn else np.nan,
        "Specificity": tn / (tn + fp) if tn + fp else np.nan,
        "AUROC": roc_auc_score(y, p),
        "Brier": brier_score_loss(y, p)}

In [ ]:
def regression_metrics(d):
    y_true = d["y_true"].astype(float).to_numpy()
    y_pred = d["y_pred"].astype(float).to_numpy()
    return {
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MSE": mean_squared_error(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred)
    }

In [ ]:
seed_results = []
for (model, seed), d in df_seeds.groupby(["Experiment", "Seed"]):
    if task=="BIN":
        metrics = binary_metrics(d, tau=0.5)
    else:
        metrics = regression_metrics(d)
    seed_results.append({"Model": model, "Seed": seed, **metrics})
df_seed_results = pd.DataFrame(seed_results)
print(df_seed_results)

In [ ]:
if task == "BIN":
    metric_cols = ["Balanced_Accuracy", "Sensitivity", "Specificity", "AUROC", "Brier"]
else:
    metric_cols = ["RMSE", "MSE", "MAE"]

df_seed_summary = (
    df_seed_results.groupby("Model")[metric_cols]
    .agg(["mean", "std"])
    .round(4))
print(df_seed_summary)
#df_seed_results.to_excel(f"/workspace/app/planilhas/TaCoLa/Seeds/{task}_SEED_RESULTS.xlsx", index=False)
#df_seed_summary.to_excel(f"/workspace/app/planilhas/TaCoLa/Seeds/{task}_SEED_SUMMARY.xlsx")

___

Paired Bootstrap using different seeds

In [ ]:
def calculate_binary_metric(y, p, metric, tau=0.5):
    y = np.asarray(y, dtype=int)
    p = np.asarray(p, dtype=float)
    pred = (p >= tau).astype(int)
    if metric == "Balanced_Accuracy":
        return balanced_accuracy_score(y, pred)
    #if metric == "Macro_F1":
    #    return f1_score(y, pred, average="macro", zero_division=0)
    if metric == "Sensitivity":
        return recall_score(y, pred, pos_label=1, zero_division=0)
    if metric == "Specificity":
        return recall_score(y, pred, pos_label=0, zero_division=0)
    if metric == "AUROC":
        return roc_auc_score(y, p) if np.unique(y).size == 2 else np.nan
    if metric == "Brier":
        return brier_score_loss(y, p)
    raise ValueError(f"Unknown metric: {metric}")

In [ ]:
def calculate_regression_metric(y, pred, metric):
    y = np.asarray(y, dtype=float)
    pred = np.asarray(pred, dtype=float)
    if metric == "RMSE":
        return np.sqrt(mean_squared_error(y, pred))
    if metric == "MSE":
        return mean_squared_error(y, pred)
    if metric == "MAE":
        return mean_absolute_error(y, pred)
    raise ValueError(f"Unknown metric: {metric}")

In [ ]:
def multiseed_paired_bootstrap(df, model_ref, model_base, metric, task="BIN", n_boot=5000, bootstrap_seed=42, tau=0.5):
    paired_by_seed = {}
    if task == "BIN":
        target_col, pred_col = "y_patient_true", "p_patient"
    else:
        target_col, pred_col = "y_true", "y_pred"

    for seed in sorted(df["Seed"].unique()):
        cols = ["Patient_ID", target_col, pred_col]
        ref = df[(df["Experiment"] == model_ref) & (df["Seed"] == seed)][cols].drop_duplicates("Patient_ID")
        base = df[(df["Experiment"] == model_base) & (df["Seed"] == seed)][cols].drop_duplicates("Patient_ID")
        paired = ref.merge(base, on="Patient_ID", suffixes=("_ref", "_base")).set_index("Patient_ID")

        if not np.allclose(paired[f"{target_col}_ref"].to_numpy(dtype=float), paired[f"{target_col}_base"].to_numpy(dtype=float)):
            raise ValueError(f"Target mismatch at seed {seed}")

        paired_by_seed[seed] = paired

    common_ids = set.intersection(*[set(paired.index) for paired in paired_by_seed.values()])
    common_ids = np.array(sorted(common_ids))
    n_patients = len(common_ids)

    def mean_seed_difference(sampled_ids):
        differences = []

        for paired in paired_by_seed.values():
            sample = paired.loc[sampled_ids]
            y = sample[f"{target_col}_ref"]
            if task == "BIN":
                ref_score = calculate_binary_metric(y, sample[f"{pred_col}_ref"], metric, tau)
                base_score = calculate_binary_metric(y, sample[f"{pred_col}_base"], metric, tau)
                difference = base_score - ref_score if metric == "Brier" else ref_score - base_score
            else:
                ref_score = calculate_regression_metric(y, sample[f"{pred_col}_ref"], metric)
                base_score = calculate_regression_metric(y, sample[f"{pred_col}_base"], metric)
                difference = base_score - ref_score

            differences.append(difference)

        return np.nanmean(differences)

    observed_difference = mean_seed_difference(common_ids)
    rng = np.random.default_rng(bootstrap_seed)
    bootstrap_differences = []

    for _ in range(n_boot):
        sampled_ids = rng.choice(common_ids, size=n_patients, replace=True)
        difference = mean_seed_difference(sampled_ids)
        if np.isfinite(difference):
            bootstrap_differences.append(difference)

    ci_low, ci_high = np.percentile(bootstrap_differences, [2.5, 97.5])
    return {
        "Reference": model_ref,
        "Baseline": model_base,
        "Metric": metric,
        "N_Patients": n_patients,
        "N_Seeds": len(paired_by_seed),
        "Difference": observed_difference,
        "CI_Lower": ci_low,
        "CI_Upper": ci_high
    }

In [ ]:
bootstrap_results = []
metrics = (
    ["Balanced_Accuracy", "Sensitivity", "Specificity", "AUROC", "Brier"]
    if task == "BIN"
    else ["RMSE", "MAE"]
)

for base in baselines:
    for metric in metrics:
        result = multiseed_paired_bootstrap(
            df_seeds, model_ref, base, metric, task=task,
            n_boot=5000, bootstrap_seed=42, tau=0.5
        )
        bootstrap_results.append(result)

df_bootstrap_seeds = pd.DataFrame(bootstrap_results)
print(df_bootstrap_seeds)

# df_bootstrap_seeds.to_excel(
#     f"/workspace/app/planilhas/TaCoLa/Seeds/{task}_MULTISEED_BOOTSTRAP.xlsx",
#     index=False
# )

___

McNemar

Compares their discordant patient-level classification errors.

In [ ]:
def paired_predictions(df, model_ref, model_base, model_col="Experiment"):
    ref = df[df[model_col] == model_ref][["Patient_ID", "y_patient_true", "y_patient_pred", "p_patient"]].drop_duplicates("Patient_ID")
    base = df[df[model_col] == model_base][["Patient_ID", "y_patient_true", "y_patient_pred", "p_patient"]].drop_duplicates("Patient_ID")

    ref = ref.rename(columns={
        "y_patient_true": "y_true_ref",
        "y_patient_pred": "y_pred_ref",
        "p_patient": "y_prob_ref"})
    base = base.rename(columns={
        "y_patient_true": "y_true_base",
        "y_patient_pred": "y_pred_base",
        "p_patient": "y_prob_base"})

    paired = ref.merge(base, on="Patient_ID", how="inner")
    if not np.array_equal(paired["y_true_ref"].to_numpy(), paired["y_true_base"].to_numpy()):
        raise ValueError("Target mismatch between models.")
    return paired

In [ ]:
def exact_mcnemar(paired, tau=0.5):
    y_true = paired["y_true_ref"].astype(int).values
    ref_pred = (paired["y_prob_ref"].astype(float).values >= tau).astype(int)
    base_pred = (paired["y_prob_base"].astype(float).values >= tau).astype(int)
    ref_correct = ref_pred == y_true
    base_correct = base_pred == y_true

    ref_only = int(np.sum(ref_correct & ~base_correct))
    base_only = int(np.sum(~ref_correct & base_correct))
    discordant = ref_only + base_only

    if discordant == 0:
        p_value = 1.0
    else:
        p_value = binomtest(ref_only, n=discordant, p=0.5, alternative="two-sided").pvalue

    return {
        "Ref_Correct_Base_Wrong": ref_only,
        "Ref_Wrong_Base_Correct": base_only,
        "Discordant_Pairs": discordant,
        "McNemar_p": p_value
    }

In [ ]:
mcnemar_results = []
for seed in [42, 123, 2026]:
    df_seed = df_seeds[df_seeds["Seed"] == seed].copy()
    for base in baselines:
        paired = paired_predictions(df_seed, model_ref, base, model_col="Experiment")
        result = exact_mcnemar(paired, tau=0.5)
        mcnemar_results.append({"Seed": seed, "Reference": model_ref, "Baseline": base, **result})
df_mcnemar_seeds = pd.DataFrame(mcnemar_results)
print(df_mcnemar_seeds)

In [ ]:
reject, adjusted_p, _, _ = multipletests(df_mcnemar_seeds["McNemar_p"].values, method="holm")
df_mcnemar_seeds["McNemar_Reject_Holm"] = reject
df_mcnemar_seeds["McNemar_p_Holm"] = adjusted_p

print(df_mcnemar_seeds)

#df_mcnemar_seeds.to_excel("/workspace/app/planilhas/TaCoLa/Seeds/BIN_MCNEMAR_SEEDS.xlsx", index=False)

___

Tau

Threshold sensitivity: The decision-threshold analysis illustrates the sensitivity–specificity trade-off across different values of tau

In [ ]:
def analyze_thresholds_multiseed(df, model_name, fixed_tau=0.5, n_taus=37):
    d = df[df["Experiment"] == model_name].copy()
    d["y_patient_true"] = pd.to_numeric(d["y_patient_true"], errors="coerce")
    d["p_patient"] = pd.to_numeric(d["p_patient"], errors="coerce")
    d = d.dropna(subset=["Seed", "Patient_ID", "y_patient_true", "p_patient"])
    d = d.drop_duplicates(["Seed", "Patient_ID"])

    taus = np.linspace(0.05, 0.95, n_taus)
    curve_rows, fixed_rows = [], []

    def calculate(y, p, tau):
        pred = (p >= tau).astype(int)
        tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
        sensitivity = tp / (tp + fn) if tp + fn else np.nan
        specificity = tn / (tn + fp) if tn + fp else np.nan
        return sensitivity, specificity

    for seed, seed_df in d.groupby("Seed"):
        y = seed_df["y_patient_true"].astype(int).to_numpy()
        p = seed_df["p_patient"].astype(float).to_numpy()
        sensitivity, specificity = calculate(y, p, fixed_tau)
        fixed_rows.append({"Model": model_name, "Seed": seed, "Tau": fixed_tau, "Sensitivity": sensitivity, "Specificity": specificity})
        for tau in taus:
            sensitivity, specificity = calculate(y, p, tau)
            curve_rows.append({"Model": model_name, "Seed": seed, "Tau": tau, "Sensitivity": sensitivity, "Specificity": specificity})

    fixed_by_seed = pd.DataFrame(fixed_rows)
    curve_by_seed = pd.DataFrame(curve_rows)

    fixed_summary = (fixed_by_seed[["Sensitivity", "Specificity"]].agg(["mean", "std"]).round(3))
    curve_summary = (curve_by_seed.groupby("Tau")[["Sensitivity", "Specificity"]].agg(["mean", "std"]).reset_index())
    curve_summary.columns = ["Tau", "Sensitivity_Mean", "Sensitivity_SD", "Specificity_Mean", "Specificity_SD"]

    fig, ax = plt.subplots(figsize=(5.0, 4.0), dpi=300)
    tau = curve_summary["Tau"].to_numpy()

    for metric, color in [("Sensitivity", "#2878B5"), ("Specificity", "#E87500")]:
        mean = curve_summary[f"{metric}_Mean"].to_numpy()
        sd = curve_summary[f"{metric}_SD"].fillna(0).to_numpy()
        ax.plot(tau, mean, color=color, linewidth=2, label=metric)
        ax.fill_between(tau, np.clip(mean - sd, 0, 1), np.clip(mean + sd, 0, 1), color=color, alpha=0.18)

    ax.axvline(fixed_tau, color="black", linestyle="--", linewidth=1.3, label=rf"Fixed $\tau={fixed_tau}$")
    ax.set_xlabel(r"Decision threshold ($\tau$)")
    ax.set_ylabel("Performance")
    ax.set_ylim(0, 1)
    ax.legend(frameon=False)
    plt.tight_layout()
    plt.show()

    return fixed_by_seed, fixed_summary, curve_by_seed, curve_summary

In [ ]:
fixed_by_seed, fixed_summary, curve_by_seed, curve_summary = (
    analyze_thresholds_multiseed(df_seeds, model_name="COLA_Alex_LogMel_Trait", fixed_tau=0.5)
    )
print(fixed_by_seed)
print(fixed_summary)

___

Panel A: Average attention per session in the lower and upper tertiles of alexithymia.

Panel B: Standardized alexithymia × temporal focus of attention, color-coded by outcome

attention seed 0

attention seed 1      

attention seed 2

=

average per patient/session → panels A and B

In [ ]:
def parse_list(value):
    if isinstance(value, (list, np.ndarray)):
        return list(value)
    if pd.isna(value):
        return []
    return list(ast.literal_eval(value))

In [ ]:
def prepare_multiseed_attention(df, model_name):
    d = df[df["Experiment"] == model_name].copy()
    d["Sessions"] = d["Sessions"].apply(parse_list)
    d["Session_Attention"] = d["Session_Attention"].apply(parse_list)
    n_seeds = d["Seed"].nunique()
    complete_patients = (d.groupby("Patient_ID")["Seed"].nunique().loc[lambda x: x == n_seeds].index)
    d = d[d["Patient_ID"].isin(complete_patients)].copy()

    rows = []
    for _, row in d.iterrows():
        if len(row["Sessions"]) != len(row["Session_Attention"]):
            continue
        for session, attention in zip(row["Sessions"], row["Session_Attention"]):
            rows.append({
                "Patient_ID": row["Patient_ID"],
                "Seed": row["Seed"],
                "Session": int(session),
                "Attention": float(attention)})

    seed_attention = pd.DataFrame(rows)
    attention_counts = (seed_attention.groupby(["Patient_ID", "Session"])["Seed"].nunique())
    if not (attention_counts == n_seeds).all():
        raise ValueError("Some patient/session attention weights are missing across seeds.")
        
    # Average the three attention weights for each patient and session.
    mean_attention = (seed_attention.groupby(["Patient_ID", "Session"], as_index=False)["Attention"].mean())

    # Renormalize across sessions within each patient.
    attention_sum = mean_attention.groupby("Patient_ID")["Attention"].transform("sum")
    mean_attention["Attention"] /= attention_sum.clip(lower=1e-8)

    patient_data = (d.groupby("Patient_ID", as_index=False).agg(Alexithymia=("Alexithymia_Standardized_Fold", "mean"), y_patient_true=("y_patient_true", "first")))
    q_low, q_high = patient_data["Alexithymia"].quantile([1/3, 2/3])
    patient_data["Alexithymia_Group"] = np.select([patient_data["Alexithymia"] <= q_low, patient_data["Alexithymia"] >= q_high], ["Low alexithymia", "High alexithymia"], default="Middle tercile")
    patient_data["Outcome"] = np.where(patient_data["y_patient_true"].astype(int) == 1, "Better", "Worse")

    session_min = mean_attention["Session"].min()
    session_max = mean_attention["Session"].max()
    denominator = max(1, session_max - session_min)

    mean_attention["Temporal_Position"] = (
        mean_attention["Session"] - session_min
    ) / denominator
    mean_attention["Weighted_Position"] = (
        mean_attention["Attention"] *
        mean_attention["Temporal_Position"])

    centers = (mean_attention.groupby("Patient_ID", as_index=False).agg(Temporal_Attention_Center=("Weighted_Position", "sum")))
    patient_data = patient_data.merge(centers, on="Patient_ID", how="inner")
    mean_attention = mean_attention.merge(patient_data[["Patient_ID", "Alexithymia", "Alexithymia_Group", "Outcome"]], on="Patient_ID", how="inner")
    return patient_data, mean_attention

In [ ]:
def bootstrap_spearman(x, y, n_boot=5000, seed=42):
    x, y = np.asarray(x, float), np.asarray(y, float)
    rho, p_value = spearmanr(x, y)
    rng = np.random.default_rng(seed)
    boot_rho = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(x), len(x))
        if np.unique(x[idx]).size < 2 or np.unique(y[idx]).size < 2:
            continue
        value = spearmanr(x[idx], y[idx]).statistic
        if np.isfinite(value):
            boot_rho.append(value)
    ci_low, ci_high = np.percentile(boot_rho, [2.5, 97.5])
    return rho, ci_low, ci_high, p_value

Interpretation of the Y-axis in Panel B:
* 0: attention focused on the first sessions;
* 0.5: attention focused approximately in the middle of the follow-up period;
* 1: attention focused on the last sessions.

How to interpret the result:
* P-value > 0: higher alexithymia is associated with relatively later attention.
* P-value <0: higher alexithymia is associated with relatively earlier attention.
* 95% CI including zero: the observed direction remains uncertain.
* The p-value tests the null hypothesis of no monotonic association.

In [ ]:
patient_attention, session_attention = prepare_multiseed_attention(df_seeds, model_name="COLA_Alex_LogMel_Trait")
rho, ci_low, ci_high, p_value = bootstrap_spearman(patient_attention["Alexithymia"], patient_attention["Temporal_Attention_Center"])
extreme_groups = session_attention[session_attention["Alexithymia_Group"].isin(["Low alexithymia", "High alexithymia"])].copy()

group_palette = {
    "High alexithymia": "#D95319",
    "Low alexithymia": "#2878B5"}
outcome_palette = {
    "Better": "#2A9D8F",
    "Worse": "#D95B69"}

fig, axes = plt.subplots(1, 2, figsize=(10, 4), dpi=300)
sns.lineplot(
    data=extreme_groups,
    x="Session", y="Attention",
    hue="Alexithymia_Group",
    hue_order=["High alexithymia", "Low alexithymia"],
    marker="o", errorbar=("ci", 95),
    palette=group_palette, ax=axes[0]
)
axes[0].set_xlabel("Session")
axes[0].set_ylabel("Mean temporal attention")
axes[0].set_title("(A) Attention profile by alexithymia")
axes[0].legend(title=None, frameon=False)

sns.scatterplot(
    data=patient_attention,
    x="Alexithymia",
    y="Temporal_Attention_Center",
    hue="Outcome",
    hue_order=["Better", "Worse"],
    palette=outcome_palette,
    s=55, alpha=0.85,
    ax=axes[1]
)
axes[1].text(
    0.04, 0.96,
    rf"$\rho_s$ = {rho:.2f}" + "\n" +
    f"95% CI [{ci_low:.2f}, {ci_high:.2f}]\n" +
    f"p = {p_value:.3f}",
    transform=axes[1].transAxes,
    va="top"
)
axes[1].set_xlabel("Standardized baseline alexithymia")
axes[1].set_ylabel("Temporal center of attention")
axes[1].set_title("(B) Alexithymia and attention timing")
axes[1].legend(title="Outcome", frameon=False)

plt.tight_layout()
plt.show()